# Notebook 9: Audit of the MIMIC-IV diagnosis codes: GEM artefact and rare-disease codes

Uses `diagnoses_icd`, `patients` and `d_icd_diagnoses` from MIMIC-IV `hosp`. Only aggregate counts are printed;
individual codes are listed only when at least 100 patients carry them.

1. Size of the GEM artefact found in Notebook 4 on the actual diagnosis rows, and its effect on label counts.
2. A corrected mapping: combination entries map to the union of their targets; for lists of alternatives the
   target most frequently coded in the ICD-9 era of MIMIC-IV is used.
3. Rare-disease codes in MIMIC-IV: ICD codes that Mondo marks as *equivalent* to a rare disease (broad and
   strict definitions, see Notebook 4), how many patients carry them, which labels they fall into, and what
   share of each label's patients they represent.
4. Rarity of labels versus rare-disease content. Without the `admissions` table the exact Notebook 1 cohort and
   split cannot be rebuilt here; label frequencies are approximated by the number of patients with at least two
   admissions who carry the label in any admission. This approximation is stated wherever it is used.

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path(os.environ.get('RAREPRED_ROOT', '..')).resolve()
sys.path.insert(0, str(ROOT / 'src'))
PROCESSED   = Path(os.environ.get('RAREPRED_PROCESSED', ROOT / 'processed'))
EXTERNAL    = ROOT / 'data' / 'external'
RESULTS     = ROOT / 'results'
FIGURES     = ROOT / 'figures'
CHECKPOINTS = ROOT / 'checkpoints'
for d in (RESULTS, FIGURES, CHECKPOINTS):
    d.mkdir(parents=True, exist_ok=True)

import collections
import numpy as np, pandas as pd
from rarepred.resources import (load_multilevel_ccs_from_rda, load_multilevel_ccs_from_hcup_csv, ccs_label,
                                load_gem_first, parse_mondo)
HOSP = ROOT / 'data' / 'hosp'
MIN_SHOW = 100   # only list codes carried by at least this many patients

import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 200, 'font.size': 9, 'axes.titlesize': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#666666', 'axes.labelcolor': '#191919', 'xtick.color': '#666666',
    'ytick.color': '#666666', 'axes.grid': True, 'grid.color': '#e6e6e6', 'grid.linewidth': 0.6,
    'axes.axisbelow': True, 'legend.frameon': False,
})
BLUE, ORANGE, AQUA, GREY = '#257de0', '#f47230', '#00b894', '#9c9c9c'

## 1. Data

In [2]:
dx = pd.read_csv(HOSP / 'diagnoses_icd.csv.gz', dtype={'icd_code': str, 'icd_version': int})
dx['icd_code'] = dx.icd_code.str.strip()
pat = pd.read_csv(HOSP / 'patients.csv.gz', usecols=['subject_id', 'anchor_age'])
titles = pd.read_csv(HOSP / 'd_icd_diagnoses.csv.gz', dtype={'icd_code': str})
title = dict(zip(zip(titles.icd_code.str.strip(), titles.icd_version), titles.long_title))
print(f"diagnosis rows {len(dx):,} (Notebook 1: 6,364,488) | ICD-9 {(dx.icd_version == 9).sum():,} | ICD-10 {(dx.icd_version == 10).sum():,}")
print(f"patients {len(pat):,} (Notebook 1: 364,627), adults {(pat.anchor_age >= 18).sum():,}")
adm_per = dx.groupby('subject_id').hadm_id.nunique()
eligible = set(adm_per[adm_per >= 2].index) & set(pat.subject_id[pat.anchor_age >= 18])
print(f"patients with >= 2 admissions carrying a diagnosis: {len(eligible):,} (Notebook 1 cohort, from admissions: 100,163)")

diagnosis rows 6,364,488 (Notebook 1: 6,364,488) | ICD-9 2,908,741 | ICD-10 3,455,747
patients 364,627 (Notebook 1: 364,627), adults 364,627


patients with >= 2 admissions carrying a diagnosis: 100,057 (Notebook 1 cohort, from admissions: 100,163)


## 2. The GEM artefact on the diagnosis rows

In [3]:
HCUP_CSV = ROOT / 'data' / 'mappings' / 'Multi_Level_CCS_2015' / 'ccs_multi_dx_tool_2015.csv'
ccs, ccs_labels = (load_multilevel_ccs_from_hcup_csv(HCUP_CSV) if HCUP_CSV.exists()
                   else load_multilevel_ccs_from_rda(EXTERNAL))
icd9_to_ccs = dict(zip(ccs.icd9, ccs.ccs_ext))
gem_first = load_gem_first(EXTERNAL / '2018_I10gem.txt')
gem = pd.read_csv(EXTERNAL / '2018_I10gem.txt', sep=r'\s+', header=None, names=['icd10', 'icd9', 'flag'], dtype=str)
gem['combination'] = gem.flag.str[2] == '1'

# Notebook 1 rule
is10 = dx.icd_version == 10
dx['icd9_nb1'] = np.where(is10, dx.icd_code.map(gem_first), dx.icd_code)
dx['ccs_nb1'] = dx.icd9_nb1.map(icd9_to_ccs)
print(f"rows with a CCS label (Notebook 1 rule): {dx.ccs_nb1.notna().mean():.4f}")

for c in ['7.1.1', '7.1.2.2']:
    s = dx[dx.ccs_nb1 == c]
    n = s.groupby('icd_version').subject_id.nunique()
    print(f"CCS {c:8s} ({ccs_label(c, ccs_labels)}): patients from ICD-9 rows {n.get(9, 0):,}, from ICD-10 rows {n.get(10, 0):,}")
i10 = dx[is10 & (dx.icd_code == 'I10')]
print(f"I10 rows {len(i10):,}, patients {i10.subject_id.nunique():,}")
print('ICD-9-era coding of essential hypertension:',
      dx[~is10 & dx.icd_code.isin(['4010', '4011', '4019'])].icd_code.value_counts().to_dict())

rows with a CCS label (Notebook 1 rule): 0.9825
CCS 7.1.1    (Essential hypertension): patients from ICD-9 rows 53,401, from ICD-10 rows 1,915


CCS 7.1.2.2  (Other hypertensive complications): patients from ICD-9 rows 543, from ICD-10 rows 48,727
I10 rows 83,775, patients 48,611


ICD-9-era coding of essential hypertension: {'4019': 102368, '4011': 3658, '4010': 397}


Under the Notebook 1 rule, essential hypertension is labelled 7.1.1 in the ICD-9 era and 7.1.2.2 in the ICD-10
era. In the ICD-9 era, malignant essential hypertension (401.0), the target chosen for I10, is coded in only a
few hundred rows.

## 3. A corrected mapping

In [4]:
icd9_era_freq = dx[~is10].icd_code.value_counts()
def corrected_targets(df):
    t = list(df.icd9)
    if df.combination.any():
        return [x for x in dict.fromkeys(t) if x != 'NoDx']        # union of the components
    t = [x for x in t if x != 'NoDx']
    if not t:
        return []
    return [max(t, key=lambda x: icd9_era_freq.get(x, 0))]         # most frequent alternative in ICD-9 era
codes10 = set(dx.icd_code[is10])
corr = {k: corrected_targets(g) for k, g in gem[gem.icd10.isin(codes10)].groupby('icd10')}

rows10 = dx[is10][['subject_id', 'hadm_id', 'icd_code']].copy()
rows10['ccs'] = rows10.icd_code.map(lambda c: [icd9_to_ccs.get(x) for x in corr.get(c, []) if icd9_to_ccs.get(x)])
rows10 = rows10.explode('ccs').dropna(subset=['ccs'])
rows9 = dx[~is10][['subject_id', 'hadm_id']].assign(ccs=dx.loc[~is10, 'ccs_nb1']).dropna(subset=['ccs'])
dx_corr = pd.concat([rows9, rows10])

def label_patients(df, col):
    d = df[df.subject_id.isin(eligible)]
    return d.groupby(col).subject_id.nunique()
n_nb1 = label_patients(dx, 'ccs_nb1')
n_cor = label_patients(dx_corr, 'ccs')
cmp_ = pd.DataFrame({'patients (Notebook 1 rule)': n_nb1, 'patients (corrected)': n_cor}).fillna(0).astype(int)
cmp_['ratio'] = (cmp_['patients (corrected)'] / cmp_['patients (Notebook 1 rule)'].replace(0, np.nan)).round(2)
changed = cmp_[(cmp_.ratio - 1).abs() >= 0.25]
print(f"labels whose patient count changes by >= 25%: {len(changed)} of {len(cmp_)}")
show = changed[changed[['patients (Notebook 1 rule)', 'patients (corrected)']].max(axis=1) >= MIN_SHOW].copy()
show['label'] = [ccs_label(c, ccs_labels) for c in show.index]
show = show.sort_values('ratio')
pd.concat([show.head(8), show.tail(6)])

labels whose patient count changes by >= 25%: 30 of 576


,patients (Notebook 1 rule),patients (corrected),ratio,label
15.7,1957,1,0.00,Other perinatal conditions
7.3.1.3,161,2,0.01,Acute but ill-defined cerebrovascular accident
7.1.2.2,29159,582,0.02,Other hypertensive complications
6.1.2,4777,359,0.08,Encephalitis (except that caused by TB or STD)
8.6.2,11089,3094,0.28,Other respiratory insufficiency
8.2.3,8327,3198,0.38,Obstructive chronic bronchitis
7.2.9.6,4666,1896,0.41,Sinoatrial node dysfunction
8.2.4,2765,1629,0.59,Other chronic pulmonary disease
7.2.11,9826,19433,1.98,Congestive heart failure; nonhypertensive
8.2.2,5331,10615,1.99,Chronic airway obstruction; not otherwise spec...


In [5]:
# which ICD-10 codes drive the largest changes, and what the GEM offers for them
d10 = dx[is10 & dx.subject_id.isin(eligible)]
for c in show.index[:4]:
    drv = d10[d10.ccs_nb1 == c].groupby('icd_code').subject_id.nunique().sort_values(ascending=False).head(1)
    for code, n in drv.items():
        t = list(gem.loc[gem.icd10 == code, 'icd9'])
        print(f"{c} <- {code} ({title.get((code, 10), '')[:45]}), {n:,} patients; GEM targets "
              + ', '.join(f"{x} [{icd9_to_ccs.get(x)}: {title.get((x, 9), '')[:35]}]" for x in t))

15.7 <- R7881 (Bacteremia), 1,956 patients; GEM targets 77183 [15.7: Bacteremia of newborn], 7907 [1.1.2.6: Bacteremia]
7.3.1.3 <- I6789 (Other cerebrovascular disease), 159 patients; GEM targets 436 [7.3.1.3: Acute, but ill-defined, cerebrovasc], 4371 [7.3.3: Other generalized ischemic cerebrov], 4378 [7.3.3: Other ill-defined cerebrovascular d]
7.1.2.2 <- I10 (Essential (primary) hypertension), 28,773 patients; GEM targets 4010 [7.1.2.2: Malignant essential hypertension], 4011 [7.1.1: Benign essential hypertension], 4019 [7.1.1: Unspecified essential hypertension]


6.1.2 <- G92 (Toxic encephalopathy), 4,471 patients; GEM targets 32371 [6.1.2: Toxic encephalitis and encephalomye], 32372 [6.1.2: Toxic myelitis], 34982 [6.9.2: Toxic encephalopathy]


## 4. Rare-disease codes in MIMIC-IV

In [6]:
mondo = parse_mondo(EXTERNAL / 'mondo' / 'src' / 'ontology' / 'mondo-edit.obo')
defs = {'broad (Mondo rare)': 'rare', 'strict (Orphanet rare)': 'orphanet_rare'}
dx['key10'] = np.where(is10, dx.icd_code, None)
dx['key9'] = np.where(~is10, dx.icd_code, None)
flags, summary = {}, []
for name, subset in defs.items():
    S = [t for t in mondo.values() if subset in t.subsets]
    c10 = {c.replace('.', '') for t in S for c in t.icd10cm_exact}
    c9 = {c for t in S for c in t.icd9_exact}
    f = dx.key10.isin(c10) | dx.key9.isin(c9)
    flags[name] = f
    pts = dx.loc[f, 'subject_id'].unique()
    summary.append({'definition': name, 'exact ICD-10-CM / ICD-9 codes': f"{len(c10)} / {len(c9)}",
                    'codes seen in MIMIC-IV': dx.loc[f, 'icd_code'].nunique(), 'rows': int(f.sum()),
                    'admissions': dx.loc[f, 'hadm_id'].nunique(), 'patients': len(pts),
                    '% of patients with a diagnosis': round(100 * len(pts) / dx.subject_id.nunique(), 2),
                    'of which >= 2 admissions': len(set(pts) & eligible)})
pd.DataFrame(summary).set_index('definition')

,exact ICD-10-CM / ICD-9 codes,codes seen in MIMIC-IV,rows,admissions,patients,% of patients with a diagnosis,of which >= 2 admissions
definition,,,,,,,
broad (Mondo rare),1025 / 682,899,94699,81890,46683,20.91,31300
strict (Orphanet rare),522 / 241,402,26201,24717,14664,6.57,10077


In [7]:
f = flags['strict (Orphanet rare)']
top = (dx[f].groupby(['icd_code', 'icd_version']).subject_id.nunique().sort_values(ascending=False))
top = top[top >= MIN_SHOW].head(15).rename('patients').reset_index()
top['title'] = [title.get((c, v), '')[:70] for c, v in zip(top.icd_code, top.icd_version)]
top

,icd_code,icd_version,patients,title
0,R570,10,2039,Cardiogenic shock
1,D473,10,1198,Essential (hemorrhagic) thrombocythemia
2,Q211,10,1129,Atrial septal defect
3,23871,9,901,Essential thrombocythemia
4,7100,9,680,Systemic lupus erythematosus
5,I5181,10,496,Takotsubo syndrome
6,D693,10,469,Immune thrombocytopenic purpura
7,I680,10,451,Cerebral amyloid angiopathy
8,D6861,10,279,Antiphospholipid syndrome
9,G500,10,263,Trigeminal neuralgia


Even the strict definition is dominated by conditions that Orphanet lists as rare but that are not unusual in a
tertiary hospital (cardiogenic shock, essential thrombocythaemia, atrial septal defect, systemic lupus
erythematosus). The long tail of
genuinely uncommon disorders is present but carried by few patients each.

## 5. Which labels carry rare-disease codes, and how rare are those labels?

In [8]:
lab_pat = dx[dx.subject_id.isin(eligible)].groupby('ccs_nb1').subject_id.nunique()
q25, q50, q75 = np.percentile(lab_pat.values, [25, 50, 75])
quart = pd.cut(lab_pat, [-np.inf, q25, q50, q75, np.inf], labels=['Q1 (rarest)', 'Q2', 'Q3', 'Q4 (common)'])
print(f"approximate label frequencies (eligible patients, any admission): {len(lab_pat)} labels, "
      f"Q25 {q25:.0f}, Q50 {q50:.0f}, Q75 {q75:.0f}")
out = []
for name, f in flags.items():
    d = dx[f & dx.subject_id.isin(eligible)]
    rp = d.groupby('ccs_nb1').subject_id.nunique()
    share = (rp / lab_pat.reindex(rp.index)).rename('share')
    t = pd.DataFrame({'quartile': quart, 'rare_patients': rp.reindex(lab_pat.index).fillna(0),
                      'share': share.reindex(lab_pat.index).fillna(0)})
    g = t.groupby('quartile', observed=False).agg(labels=('rare_patients', 'size'),
                                                  labels_with_rare_code=('rare_patients', lambda s: int((s > 0).sum())),
                                                  rare_patients=('rare_patients', 'sum'),
                                                  median_share_where_present=('share', lambda s: float(s[s > 0].median()) if (s > 0).any() else np.nan))
    g.insert(0, 'definition', name)
    out.append(g)
by_quartile = pd.concat(out).round(3)
by_quartile

approximate label frequencies (eligible patients, any admission): 576 labels, Q25 311, Q50 1487, Q75 4491


,definition,labels,labels_with_rare_code,rare_patients,median_share_where_present
quartile,,,,,
Q1 (rarest),broad (Mondo rare),144,40,784.0,0.168
Q2,broad (Mondo rare),144,35,6744.0,0.206
Q3,broad (Mondo rare),144,41,11783.0,0.033
Q4 (common),broad (Mondo rare),144,51,24961.0,0.012
Q1 (rarest),strict (Orphanet rare),144,28,371.0,0.158
Q2,strict (Orphanet rare),144,21,2235.0,0.068
Q3,strict (Orphanet rare),144,29,3497.0,0.014
Q4 (common),strict (Orphanet rare),144,38,5090.0,0.004


In [9]:
f = flags['strict (Orphanet rare)'] & dx.subject_id.isin(eligible)
rp = dx[f].groupby('ccs_nb1').subject_id.nunique()
tab = pd.DataFrame({'rare-coded patients': rp, 'label patients': lab_pat.reindex(rp.index)})
tab['share of label'] = (tab['rare-coded patients'] / tab['label patients']).round(3)
tab['quartile'] = quart.reindex(tab.index)
tab['label'] = [ccs_label(c, ccs_labels) for c in tab.index]
tab[tab['rare-coded patients'] >= MIN_SHOW].sort_values('rare-coded patients', ascending=False).head(12)

,rare-coded patients,label patients,share of label,quartile,label
ccs_nb1,,,,,
2.14,1668,3615,0.461,Q3,Neoplasms of unspecified nature or uncertain b...
17.1.5,1254,8675,0.145,Q4 (common),Shock
13.7,774,1340,0.578,Q2,Systemic lupus erythematosus and connective ti...
14.1.4,763,1279,0.597,Q2,Atrial septal defect
4.2.2,503,14267,0.035,Q4 (common),Thrombocytopenia
7.2.7,429,2743,0.156,Q3,Other and ill-defined heart disease
7.4.4.2,373,18606,0.020,Q4 (common),Other and unspecified circulatory disease
6.9.2,353,13921,0.025,Q4 (common),Other central nervous system disorders
6.9.1,310,11113,0.028,Q4 (common),Disorders of the peripheral nervous system


## 6. Are admissions with a rare-disease code more complex? (Hypothesis B)

In [10]:
per_adm = dx.groupby('hadm_id').agg(n_dx=('icd_code', 'size'))
for name, f in flags.items():
    adm_r = set(dx.loc[f, 'hadm_id'])
    m = per_adm.index.isin(adm_r)
    print(f"{name:24s}: diagnosis codes per admission, with a rare code {per_adm.n_dx[m].median():.0f} "
          f"(mean {per_adm.n_dx[m].mean():.1f}) vs without {per_adm.n_dx[~m].median():.0f} (mean {per_adm.n_dx[~m].mean():.1f})")

broad (Mondo rare)      : diagnosis codes per admission, with a rare code 15 (mean 16.7) vs without 9 (mean 10.8)
strict (Orphanet rare)  : diagnosis codes per admission, with a rare code 16 (mean 17.1) vs without 10 (mean 11.4)


## 7. Save

In [11]:
res = {'n_rows': len(dx), 'n_eligible_patients_approx': len(eligible),
       'i10': {'rows': len(i10), 'patients': int(i10.subject_id.nunique())},
       'hypertension_split': {c: dx[dx.ccs_nb1 == c].groupby('icd_version').subject_id.nunique().to_dict() for c in ['7.1.1', '7.1.2.2']},
       'n_labels_changed_25pct': len(changed),
       'label_patients_nb1_vs_corrected': cmp_.loc[[c for c in ['7.1.1', '7.1.2.2', '15.7', '6.1.2', '8.6.1', '8.6.2', '7.2.11'] if c in cmp_.index]].to_dict(orient='index'),
       'rare_code_summary': summary,
       'label_quartiles_approx': {'q25': q25, 'q50': q50, 'q75': q75},
       'by_quartile': by_quartile.reset_index().to_dict(orient='records')}
with open(RESULTS / 'mimic_code_audit.json', 'w') as fh:
    json.dump(res, fh, indent=2, default=str)
print('saved results/mimic_code_audit.json')

saved results/mimic_code_audit.json
